# Reachable angles and measured decoder costs

**Our preceding best official result is 22.78244 (V22).** This update combines Heliosli's latest compatible angle loaders, Dmitrii Gluzdov's lower-normalization weights, and a measured comparison of two public streamed DQI policies. It also corrects our V23 interpretation of the chemistry contract.

The components are credited prominently because this is a controlled portfolio, not a claim that we invented their circuits. Our work here is the independent MPS audit, fixed-condition comparisons, rejected gauge search, exact-byte integration and readable reproduction.

| Component | Original contribution | Evidence used here |
|---|---|---|
| 63Q inverse square root | busyaprime, on mrmeows | Our unchanged full 200,540,928-shot receipt |
| 895Q streamed decoder | Mudit Chaturvedi, on SomeJoint | Our fresh full 102,400-shot comparison |
| 896Q decoder policy | Atakan Aldemir, on SomeJoint and Mudit | Our fresh full 102,400-shot comparison |
| Fifteen angle loaders | Heliosli, on SomeJoint | 73,200 actual-gate MPS trajectories and fresh 78,080-shot native check |
| Chemistry weights | Dmitrii Gluzdov | Exact public factor, checked with the same angle dictionary |

The generated table below reports **local verifier costs and predictions**. A new official Public Score exists only after competition grading. CPU only, accelerator **None**, internet off.


## 1. The official ANGLE contract

[Competition host Tanuj Khattar clarified the target-register contract](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092): circuits only need to be correct on the states reached by the specified forward and reverse traversal, at every valid address.

Our V23 arbitrary-target counterexample remains a valid test of the stronger general-XOR property. **It does not invalidate a loader that satisfies the host's reachable-state requirement.** Withholding submissions until that stronger property was repaired was an unnecessarily strict interpretation; this version corrects it.

For batch `k`, our MPS audit starts from the previous batch's words on the forward path, and the current batch's words on the reverse path. It follows all fifteen actual gate streams at every one of the 1,220 valid addresses, with an all-zero and a deterministic varying measurement stream. The 73,200 trajectories check words, phase, selection restoration and clean scratch. The original native verifier separately samples its full 78,080 shots. Neither sampled check is presented as a proof over all measurement records.

The sign-copy implementation still has the explicit `bit7 == bit6` encoded-word precondition. We do not call it a general arbitrary-target XOR oracle. Our earlier clean-cache general-XOR study remains a separate stronger-contract experiment.


## 2. Compare decoder costs without hiding failures

Mudit's 895Q policy uses ordinary external nodes 8 and 9. Atakan's 896Q policy additionally makes node 7 homogeneous, in both forward and replay namespaces. Atakan reports that this repairs two fixed logical counterexamples, including a phase case in the unmodified 895Q and 897Q policies.

Those targeted cases are useful diagnostics; they are not an unbiased failure-rate sample. The official DQI metric permits failures and includes them in its repetition penalty. We therefore show both full native comparison results, retain every observed value/phase error, and select the lower **complete measured cost** at the single declared salt `oct29-round7-independent`. There are no salt retries or discarded checks. Both policies retain their original public bytes. The original circuit-dependent input and measurement generation is retained, so this is not a claim of identical logical samples across different hashes.

A zero observed error count does not refute the reported counterexample or establish exact correctness. The selected policy is the cheaper one on this declared check, not a claim that its true error rate is universally better. Our previous independent 895Q check is also retained in the research log; it is not substituted for an unfavorable fresh result.


In [ ]:
import os

os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, tempfile, shutil, sys, math, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SALT = "oct29-round7-independent"


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


In [ ]:
SOURCE_ARCHIVE_SHA256 = "8095aa9c05efc789fa538af385829f9af6f699580aa832f5332f0214d1960396"
SOURCE_MEMBERS = {
    "dfthcsos/angle_k_0.kmb": "114a2acb164b6538af6fac7784d97ce4bca52cd59878b9ae597889f230888f71",
    "dfthcsos/angle_k_1.kmb": "a01248289fe2a488a39491455089b16bd41b8bff1e86b6dce6be5e85c5019385",
    "dfthcsos/angle_k_10.kmb": "708409dc540b0726438b94854f959a2b6641de6b1aa9b0840beaffe3862f6ba2",
    "dfthcsos/angle_k_11.kmb": "285c86ed15771b8c01a69e98a6098572fa1d923448d0a8cdfad70ebdfceefe91",
    "dfthcsos/angle_k_12.kmb": "8375a66c80b83e4a4ac697112769fbba126d6f58fc58e7591aba01fb864bd269",
    "dfthcsos/angle_k_13.kmb": "7a854def7b94ed80be187e58fc456dca90a86d2838c39e7811e8e02b2a8ba77a",
    "dfthcsos/angle_k_14.kmb": "be4bd2e0e6f63e5c37a232aec0b2a30fd693b4e928605f1d4ade6f4263ffa39c",
    "dfthcsos/angle_k_2.kmb": "b88a234ab24b00c657c4b3b5e8f3f680ac1e41208a64abad08ef4bf87f0d475e",
    "dfthcsos/angle_k_3.kmb": "abf0069e563e4797ed7e3b74d65f932aec131974009f80cb323ae5f5d754fb56",
    "dfthcsos/angle_k_4.kmb": "017d5072da349301a8c0ca5243ffd058528c38531e88728f6735b29f48a65eab",
    "dfthcsos/angle_k_5.kmb": "42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099",
    "dfthcsos/angle_k_6.kmb": "d30c71fb6a288e946f688829f56b6c7026418337441e87a0a2bb5c7bb4fab794",
    "dfthcsos/angle_k_7.kmb": "8cb1eca28096d1d07d46443bd9081a723a0ef885f7fc3c02d85abb8aa49d41a7",
    "dfthcsos/angle_k_8.kmb": "ea2d9ab05bd9c61d8e4f2a21ba20c188e08b0db614c222aafa1af0d948f93f18",
    "dfthcsos/angle_k_9.kmb": "cf971aaf982c21482d200f33324f8d6dea02686bb137223cf3d3a932b8202d17",
    "dfthcsos/submission.npz": "4f78b7b841745c42bb14d92e55a0ba5a9a035c752b9d7a715997395eb4729bdf",
    "dqi.kmb": "0ff3257b1f116acfeab2813bd256e9817fe1ad5a42c7c78db9d905f6db00c1cc",
    "invsqrt.kmb": "9a8627c812833a6feca6eb82239f38d1a9a5e1eee299a9a7af9abd85e36339be",
}
SELECTED_MEMBERS = {
    "dfthcsos/angle_k_0.kmb": "114a2acb164b6538af6fac7784d97ce4bca52cd59878b9ae597889f230888f71",
    "dfthcsos/angle_k_1.kmb": "a01248289fe2a488a39491455089b16bd41b8bff1e86b6dce6be5e85c5019385",
    "dfthcsos/angle_k_10.kmb": "708409dc540b0726438b94854f959a2b6641de6b1aa9b0840beaffe3862f6ba2",
    "dfthcsos/angle_k_11.kmb": "285c86ed15771b8c01a69e98a6098572fa1d923448d0a8cdfad70ebdfceefe91",
    "dfthcsos/angle_k_12.kmb": "8375a66c80b83e4a4ac697112769fbba126d6f58fc58e7591aba01fb864bd269",
    "dfthcsos/angle_k_13.kmb": "7a854def7b94ed80be187e58fc456dca90a86d2838c39e7811e8e02b2a8ba77a",
    "dfthcsos/angle_k_14.kmb": "be4bd2e0e6f63e5c37a232aec0b2a30fd693b4e928605f1d4ade6f4263ffa39c",
    "dfthcsos/angle_k_2.kmb": "b88a234ab24b00c657c4b3b5e8f3f680ac1e41208a64abad08ef4bf87f0d475e",
    "dfthcsos/angle_k_3.kmb": "abf0069e563e4797ed7e3b74d65f932aec131974009f80cb323ae5f5d754fb56",
    "dfthcsos/angle_k_4.kmb": "017d5072da349301a8c0ca5243ffd058528c38531e88728f6735b29f48a65eab",
    "dfthcsos/angle_k_5.kmb": "42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099",
    "dfthcsos/angle_k_6.kmb": "d30c71fb6a288e946f688829f56b6c7026418337441e87a0a2bb5c7bb4fab794",
    "dfthcsos/angle_k_7.kmb": "8cb1eca28096d1d07d46443bd9081a723a0ef885f7fc3c02d85abb8aa49d41a7",
    "dfthcsos/angle_k_8.kmb": "ea2d9ab05bd9c61d8e4f2a21ba20c188e08b0db614c222aafa1af0d948f93f18",
    "dfthcsos/angle_k_9.kmb": "cf971aaf982c21482d200f33324f8d6dea02686bb137223cf3d3a932b8202d17",
    "dfthcsos/submission.npz": "4f78b7b841745c42bb14d92e55a0ba5a9a035c752b9d7a715997395eb4729bdf",
    "dqi.kmb": "de2d17dc7760974779178af6d96f0ac2c97babdf887011b33ca8b729182a05e7",
    "invsqrt.kmb": "9a8627c812833a6feca6eb82239f38d1a9a5e1eee299a9a7af9abd85e36339be",
}
POLICY_REPORTS = {
    "895Q / Mudit": {
        "status": "PASS",
        "score": 412645903413.51733,
        "m": 4095,
        "n": 70,
        "degree": 12,
        "r": 2016,
        "verified_shots": 102400,
        "value_errors": 0,
        "phase_errors": 0,
        "empirical_error_rate": 0.0,
        "effective_error_rate": 0.0008660587301657926,
        "log2_delta": -128.0,
        "repetition_factor": 1.1912604648115832,
        "mean_error_weight": 31.046737331746634,
        "toffoli_count": 432438.8846484375,
        "max_toffoli_count": 445997,
        "reaction_depth": 8179,
        "num_ops": 60395385,
        "shot_operation_counts": {
            "NEG": 10034119,
            "BIT_INVERT": 152938751668,
            "BIT_STORE0": 12473139200,
            "BIT_STORE1": 273092888,
            "X": 25553508072,
            "Z": 3708840555,
            "R": 6609348521,
            "HMR": 10192631721,
            "CX": 5461275632316,
            "CZ": 47173976000,
            "SWAP": 166072414596,
            "CSWAP": 1408204800,
            "CCX": 42773801404,
            "CCZ": 99735584,
            "INFO_TAG": 1939865600,
            "POP_CONDITION": 1942323200,
            "PUSH_CONDITION": 1942323200,
        },
        "num_qubits": 895,
        "num_bits": 105130,
        "circuit_sha256": "de2d17dc7760974779178af6d96f0ac2c97babdf887011b33ca8b729182a05e7",
    },
    "896Q / Atakan": {
        "status": "PASS",
        "score": 413022404623.3406,
        "m": 4095,
        "n": 70,
        "degree": 12,
        "r": 2016,
        "verified_shots": 102400,
        "value_errors": 0,
        "phase_errors": 0,
        "empirical_error_rate": 0.0,
        "effective_error_rate": 0.0008660587301657926,
        "log2_delta": -128.0,
        "repetition_factor": 1.1912604648115832,
        "mean_error_weight": 31.046737331746634,
        "toffoli_count": 431867.83818359376,
        "max_toffoli_count": 445569,
        "reaction_depth": 8258,
        "num_ops": 60177766,
        "shot_operation_counts": {
            "NEG": 9317889,
            "BIT_INVERT": 151500726533,
            "BIT_STORE0": 12514918400,
            "BIT_STORE1": 281703738,
            "X": 25559236760,
            "Z": 3418823147,
            "R": 6589171558,
            "HMR": 9949427558,
            "CX": 5445431841091,
            "CZ": 45959747613,
            "SWAP": 165596948318,
            "CSWAP": 1418035200,
            "CCX": 42715736242,
            "CCZ": 89495188,
            "INFO_TAG": 1899724800,
            "POP_CONDITION": 1971916800,
            "PUSH_CONDITION": 1971916800,
        },
        "num_qubits": 896,
        "num_bits": 104480,
        "circuit_sha256": "0ff3257b1f116acfeab2813bd256e9817fe1ad5a42c7c78db9d905f6db00c1cc",
    },
}
SELECTED_POLICY = "895Q / Mudit"
MPS_RECEIPT = {
    "device": "mps",
    "status": "PASS",
    "scope": "Exact AtakanV11 / Heliosli latest 15 ANGLE streams; all1220 addresses, both reachable incoming batches, two HMR streams. Word values, phase, selection restoration and scratch checked. This is sampled branch evidence, not an all-measurement-record proof.",
    "basis_trajectories": 73200,
    "batches": [
        {
            "batch": 0,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 1,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 2,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 3,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 4,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 5,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 6,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 7,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 8,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 9,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 10,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 11,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 12,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 13,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
        {
            "batch": 14,
            "value_failures": 0,
            "phase_failures": 0,
            "first_value_counterexample": None,
        },
    ],
    "seconds": 82.32779295812361,
    "circuit_hashes": {
        "dfthcsos/angle_k_0.kmb": "114a2acb164b6538af6fac7784d97ce4bca52cd59878b9ae597889f230888f71",
        "dfthcsos/angle_k_1.kmb": "a01248289fe2a488a39491455089b16bd41b8bff1e86b6dce6be5e85c5019385",
        "dfthcsos/angle_k_10.kmb": "708409dc540b0726438b94854f959a2b6641de6b1aa9b0840beaffe3862f6ba2",
        "dfthcsos/angle_k_11.kmb": "285c86ed15771b8c01a69e98a6098572fa1d923448d0a8cdfad70ebdfceefe91",
        "dfthcsos/angle_k_12.kmb": "8375a66c80b83e4a4ac697112769fbba126d6f58fc58e7591aba01fb864bd269",
        "dfthcsos/angle_k_13.kmb": "7a854def7b94ed80be187e58fc456dca90a86d2838c39e7811e8e02b2a8ba77a",
        "dfthcsos/angle_k_14.kmb": "be4bd2e0e6f63e5c37a232aec0b2a30fd693b4e928605f1d4ade6f4263ffa39c",
        "dfthcsos/angle_k_2.kmb": "b88a234ab24b00c657c4b3b5e8f3f680ac1e41208a64abad08ef4bf87f0d475e",
        "dfthcsos/angle_k_3.kmb": "abf0069e563e4797ed7e3b74d65f932aec131974009f80cb323ae5f5d754fb56",
        "dfthcsos/angle_k_4.kmb": "017d5072da349301a8c0ca5243ffd058528c38531e88728f6735b29f48a65eab",
        "dfthcsos/angle_k_5.kmb": "42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099",
        "dfthcsos/angle_k_6.kmb": "d30c71fb6a288e946f688829f56b6c7026418337441e87a0a2bb5c7bb4fab794",
        "dfthcsos/angle_k_7.kmb": "8cb1eca28096d1d07d46443bd9081a723a0ef885f7fc3c02d85abb8aa49d41a7",
        "dfthcsos/angle_k_8.kmb": "ea2d9ab05bd9c61d8e4f2a21ba20c188e08b0db614c222aafa1af0d948f93f18",
        "dfthcsos/angle_k_9.kmb": "cf971aaf982c21482d200f33324f8d6dea02686bb137223cf3d3a932b8202d17",
    },
}
EXPECTED_CHEMISTRY = {
    "status": "PASS",
    "score": 340441129.56165034,
    "lambda_eff": 48.613751511735046,
    "toffoli_count": 16286.0,
    "num_qubits": 430,
    "brot": 8,
    "batch_size": 5,
    "num_circuits": 15,
    "reaction_depth": 165,
    "num_ops": 585310,
    "verified_shots": 78080,
    "submission_sha256": "a2c5b9e7559ad2ae3ec70f94ec2364ca522346cd6028ce76d2571c172ea78b44",
    "resources": {
        "brot": 8,
        "keep_bits": 10,
        "angles_toffoli_per_pass": 4178.0,
        "angles_toffoli": 8356.0,
        "angles_qubits": 126,
        "angles_selection_qubits": 11,
        "angles_persistent_qubits": 40,
        "angles_temporary_qubits": 75,
        "outer_state_prep_toffoli": 369,
        "inner_state_prep_toffoli": 5311,
        "givens_rotations_toffoli": 2250,
        "outer_state_prep_persistent_qubits": 42,
        "outer_state_prep_temporary_qubits": 96,
        "outer_state_prep_qubits": 138,
        "inner_state_prep_persistent_qubits": 44,
        "inner_state_prep_temporary_qubits": 304,
        "inner_state_prep_qubits": 348,
        "heuristic_persistent_qubits": 86,
        "heuristic_temporary_qubits": 304,
        "heuristic_toffoli": 7930,
        "heuristic_qubits": 390,
        "persistent_qubits": 126,
        "temporary_qubits": 304,
        "toffoli": 16286.0,
        "qubits": 430,
        "angles_within_cap": True,
    },
    "metrics": {
        "error_1": 0.019999450002491374,
        "error_2": 0.019999450000266143,
        "Lambda": 303.79117572502224,
        "E_SOS": -22144.188248550006,
        "E_gap": 3.9148985500069102,
        "lambda_eff": 48.613751511735046,
        "gap_in_domain": True,
    },
}


## 3. Pin the public inputs and authenticate them

Attach the competition, V9 of our research-assets dataset, Atakan's **V11 output (363403318)**, and Mudit's **V3 output (363366074)**. These notebook inputs are data sources: no competitor Python is executed. The exact archive and all eighteen selected members are authenticated below. A changed upstream version stops the run; it is never silently accepted as “latest.”

Atakan's frozen archive supplies the new chemistry and unchanged arithmetic. If the fixed comparison selects 895Q, only the decoder binary is replaced with Mudit's authenticated file. Original authorship remains with the named authors in either case.


In [ ]:
COMP = next(
    p.parents[2]
    for p in INPUT.rglob("verify_submission_invsqrt.py")
    if p.parents[2].name == "google-verifiable-quantum-circuit-compilation"
)
ASSETS = next(p.parent for p in INPUT.rglob("oct29_manifest.json"))
asset_manifest = json.loads((ASSETS / "oct29_manifest.json").read_text())
for name, expected in asset_manifest["official_inputs"].items():
    assert sha256(COMP / name) == expected, name
source_archive = next(
    p
    for p in INPUT.rglob("submission.zip")
    if "atakanaldemir" in str(p) and sha256(p) == SOURCE_ARCHIVE_SHA256
)
SUB = Path(tempfile.mkdtemp(prefix="reachable-angles-"))
with zipfile.ZipFile(source_archive) as archive:
    assert set(archive.namelist()) == set(SOURCE_MEMBERS)
    for name, expected in SOURCE_MEMBERS.items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and ".." not in member.parts
        blob = archive.read(name)
        assert hashlib.sha256(blob).hexdigest() == expected, name
        target = SUB / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(blob)
if SELECTED_MEMBERS["dqi.kmb"] != SOURCE_MEMBERS["dqi.kmb"]:
    decoder_source = next(
        p
        for p in INPUT.rglob("dqi.kmb")
        if p.stat().st_size == 547702270 and sha256(p) == SELECTED_MEMBERS["dqi.kmb"]
    )
    shutil.copyfile(decoder_source, SUB / "dqi.kmb")
actual = {str(p.relative_to(SUB)): sha256(p) for p in SUB.rglob("*") if p.is_file()}
assert actual == SELECTED_MEMBERS
assert MPS_RECEIPT["circuit_hashes"] == {
    k: v for k, v in actual.items() if k.startswith("dfthcsos/angle_")
}
print("Authenticated all eighteen members; selected policy:", SELECTED_POLICY)


## 4. Original verification, with explicit receipt scope

The arithmetic file is unchanged and retains our own full 200,540,928-shot receipt. Both decoder candidates have just undergone the complete unchanged official verifier, each at the declared salt and 102,400 shots. Their exact-hash receipts are printed, including any errors and the actual repetition penalty. `REPEAT_STATIC_CHECKS = True` repeats the selected static checks; repeating the same salt does not add independent evidence.

Chemistry is always checked afresh, at all 78,080 shots, against the original reference integrals, fixed shifts, accuracy bounds and energy-gap floor. No Hamiltonian inputs or verifier conditions are modified. The selected numerical factor remains close to the original acceptance boundary, so the numerical check is essential.


In [ ]:
sys.path[:0] = [str(COMP / "kickmix"), str(COMP / "verifiers")]
import kickmix as km
from invsqrt import verify_submission_invsqrt as inverse_sqrt
from dqi import verify_submission_dqi as decoder
from dfthcsos import verify as chemistry

REPEAT_STATIC_CHECKS = False
inverse_report = json.loads((ASSETS / "new_inv_full.json").read_text())
assert inverse_report["status"] == "PASS"
assert inverse_report["verified_shots"] == 200540928
assert inverse_report["circuit_sha256"] == actual["invsqrt.kmb"]
decoder_report = dict(POLICY_REPORTS[SELECTED_POLICY])
assert decoder_report["status"] == "PASS" and decoder_report["verified_shots"] == 102400
assert decoder_report["circuit_sha256"] == actual["dqi.kmb"]
if REPEAT_STATIC_CHECKS:
    circuit = km.Circuit.from_file(str(SUB / "invsqrt.kmb"), format="kmb")
    inverse_report = inverse_sqrt.verify(
        circuit,
        inverse_sqrt.ProblemSpecInvSqrt.for_n_p(10),
        salt=asset_manifest["salt"],
        max_workers=2,
    )
    del circuit
    gc.collect()
    circuit = km.Circuit.from_file(str(SUB / "dqi.kmb"), format="kmb")
    decoder_report = decoder.verify(
        circuit, decoder.ProblemSpecDQI.benchmark_instance(), salt=SALT, max_workers=2
    )
    del circuit
    gc.collect()
chemistry_report = chemistry.verify(str(SUB / "dfthcsos"), salt=SALT)
assert chemistry_report["status"] == "PASS" and chemistry_report["verified_shots"] == 78080
assert chemistry_report["submission_sha256"] == EXPECTED_CHEMISTRY["submission_sha256"]
assert math.isclose(chemistry_report["score"], EXPECTED_CHEMISTRY["score"], rel_tol=1e-12)
reports = {"invsqrt": inverse_report, "dqi": decoder_report, "dfthcsos": chemistry_report}
display(
    pd.DataFrame(
        [
            {
                "Task": task,
                "Status": r["status"],
                "Shots": r["verified_shots"],
                "Q": r["num_qubits"],
                "Cost": r["score"],
            }
            for task, r in reports.items()
        ]
    )
)


## 5. See the whole cost, not qubits alone

The DQI bill includes `Q² × mean Toffolis × repetition penalty`. A one-qubit difference is not enough to choose a policy. The table uses the same newly verified chemistry and unchanged arithmetic for both rows, so it isolates the decoder comparison. These are local measurements at the declared conditions; competition grading can observe different failures and costs.


In [ ]:
reference = asset_manifest["reference_costs"]
fixed_product = (
    reference["invsqrt"]
    / inverse_report["score"]
    * reference["dfthcsos"]
    / chemistry_report["score"]
)
comparison_rows = []
for label, r in POLICY_REPORTS.items():
    comparison_rows.append(
        {
            "Policy": label,
            "Q": r["num_qubits"],
            "Mean Toffolis": r["toffoli_count"],
            "Shots": r["verified_shots"],
            "Value errors": r["value_errors"],
            "Phase errors": r["phase_errors"],
            "Repetition factor": r["repetition_factor"],
            "DQI cost": r["score"],
            "Local GM": (fixed_product * reference["dqi"] / r["score"]) ** (1 / 3),
        }
    )
comparison = pd.DataFrame(comparison_rows)
display(comparison)
local_gm = (fixed_product * reference["dqi"] / decoder_report["score"]) ** (1 / 3)
print(f"Selected local GM: {local_gm:.8f}; not a new official Public Score.")
fig, ax = plt.subplots(figsize=(9, 3.2), layout="constrained")
ax.barh(comparison["Policy"], comparison["Local GM"], color=["#177e89", "#8b99ad"])
for i, row in comparison.iterrows():
    ax.text(row["Local GM"], i, f' {row["Local GM"]:.6f}', va="center")
ax.set(
    xlim=(comparison["Local GM"].min() - 0.003, comparison["Local GM"].max() + 0.005),
    xlabel="Local verifier geometric mean",
    title="Fixed-condition decoder comparison",
)
fig.savefig(WORK / "decoder_comparison.png", dpi=160)
plt.show()
comparison.to_csv(WORK / "decoder_comparison.csv", index=False)


## 6. Our extra search was not selected

We also tried three-column Euler kicks followed by exact pair relaxation on the new Gluzdov factor, entirely on Mac MPS. The best float32 proxy changed by only about `8.4e-5 / 1163.35` (roughly `7e-8` relative). That is below a useful, trustworthy improvement here. The numerical factor in this release is **Gluzdov's unchanged public file**; this search is not counted as an original score gain.

Our independent full GF(2²⁴) inverse audit from the preceding study covers all 16,777,216 field elements, including zero. It checks the shared finite-field algebra, not the complete decoder gate stream or its measurement-phase correction. The full native reports and the separately attributed targeted diagnostics are the relevant decoder evidence.


## 7. Export exactly the verified data

Only the two main circuit binaries, fifteen ANGLE binaries and the numerical factor enter `submission.zip`. The exact member dictionary is checked again. Receipts, the comparison and a Mac-audit input bundle are exported separately. No output from this cell is automatically submitted.


In [ ]:
assert {
    str(p.relative_to(SUB)): sha256(p) for p in SUB.rglob("*") if p.is_file()
} == SELECTED_MEMBERS
archive_path = WORK / "submission.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in sorted(SELECTED_MEMBERS):
        info = zipfile.ZipInfo(name, date_time=(2026, 10, 29, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, (SUB / name).read_bytes())
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    assert {
        name: hashlib.sha256(archive.read(name)).hexdigest() for name in archive.namelist()
    } == SELECTED_MEMBERS
receipt = {
    "archive_sha256": sha256(archive_path),
    "members": SELECTED_MEMBERS,
    "reports": reports,
    "decoder_comparison": POLICY_REPORTS,
    "salt": SALT,
    "mps_receipt": MPS_RECEIPT,
    "local_geometric_mean": local_gm,
    "official_public_score": None,
    "official_angle_clarification": "https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092",
}
(WORK / "verified_release.json").write_text(json.dumps(receipt, indent=2))
(WORK / "reachable_mps_report.json").write_text(json.dumps(MPS_RECEIPT, indent=2))
mac_inputs = WORK / "mac_audit_inputs"
mac_inputs.mkdir(exist_ok=True)
angle_texts = {}
for k in range(15):
    circuit = km.Circuit.from_file(str(SUB / f"dfthcsos/angle_k_{k}.kmb"), format="kmb")
    text_path = mac_inputs / f"angle_k_{k}.kmx"
    circuit.to_file(str(text_path), format="kmx")
    angle_texts[f"angle_k_{k}"] = text_path.read_text()
(mac_inputs / "angle_kmxs.json").write_text(json.dumps(angle_texts))
shutil.copyfile(SUB / "dfthcsos/submission.npz", mac_inputs / "submission.npz")
print("Archive SHA-256:", receipt["archive_sha256"])
display(FileLink(str(archive_path)))


## Optional: repeat the reachable-state audit on Mac MPS

Download `mac_audit_inputs` from the output, then call `run_mac_mps_audit("mac_audit_inputs")` on a Mac with PyTorch MPS. The function below is readable source and is not called by the Kaggle CPU run. It follows the actual gate stream, including measurement reset and conditioned phase correction. Two measurement streams are diagnostic samples, not an all-branches proof.


In [ ]:
def run_mac_mps_audit(data_dir):
    """MPS packed basis trajectories of the actual published J5 loaders.
    All valid addresses; zero/arbitrary XOR targets; two measurement streams.
    The complete unchanged native verifier is separate.
    """
    import os

    os.environ["OMP_NUM_THREADS"] = "1"
    import torch, numpy as np, json, re, time
    from pathlib import Path
    from collections import Counter

    torch.set_num_threads(1)
    assert torch.backends.mps.is_available()
    t = time.monotonic()
    dev = "mps"
    root = Path(data_dir)
    MASK = (1 << 63) - 1
    with np.load(root / "submission.npz") as z:
        words = torch.from_numpy(z["angle_words"].copy()).to(dev, dtype=torch.int64) & 255
    A, columns = words.shape
    N = A * 4
    W = (N + 62) // 63
    ids = torch.arange(W * 63, device=dev, dtype=torch.int64).reshape(W, 63)
    addr = ids % A
    cases = ids // A
    weights = torch.ones(63, device=dev, dtype=torch.int64) << torch.arange(
        63, device=dev, dtype=torch.int64
    )

    def pack(bits):
        return (bits * weights).sum(-1)

    sel = torch.stack([pack((addr >> j) & 1) for j in range(11)])
    stream = pack((cases >> 1) & 1)
    lanes = torch.arange(W, device=dev, dtype=torch.int64)
    FULL = torch.full((W,), MASK, device=dev, dtype=torch.int64)
    payload = json.loads((root / "angle_kmxs.json").read_text())
    records = []
    for k in range(15):
        text = payload["angle_k_" + str(k)]
        lines = text.splitlines()
        current = words[:, k * 5 : k * 5 + 5]
        previous = words[:, (k - 1) * 5 : k * 5] if k else torch.zeros_like(current)
        current = current[addr.reshape(-1)].reshape(W, 63, 5).permute(2, 0, 1)
        previous = previous[addr.reshape(-1)].reshape(W, 63, 5).permute(2, 0, 1)
        target = torch.where(((cases % 2) == 0)[None], previous, current)
        target_bits = torch.stack(
            [pack((target[j] >> b) & 1) for j in range(5) for b in range(8)]
        )
        q = torch.zeros((128, W), device=dev, dtype=torch.int64)
        q[:11] = sel
        q[11:51] = target_bits
        bits = {}
        phase = torch.zeros(W, device=dev, dtype=torch.int64)
        ordinal = 0
        ops = []
        for line in lines:
            a = line.split()
            if not a or a[0] in ["APPEND_TO_REGISTER", "REGISTER", "INFO_TAG"]:
                continue
            op = a[0]
            qs = [int(x[1:]) for x in a[1:] if re.fullmatch(r"q\d+", x)]
            cond = a[a.index("if") + 1 :] if "if" in a else []
            ops.append((op, qs, cond, a))
        i = 0
        while i < len(ops):
            op, qs, cond, a = ops[i]
            active = FULL if not cond else FULL.clone()
            for b in cond:
                active &= bits[b]
            if op == "CX" and not cond:
                control = qs[0]
                targets = [qs[1]]
                i += 1
                while (
                    i < len(ops)
                    and ops[i][0] == "CX"
                    and not ops[i][2]
                    and ops[i][1][0] == control
                ):
                    targets.append(ops[i][1][1])
                    i += 1
                targets = [v for v, n in Counter(targets).items() if n % 2]
                if targets:
                    tt = torch.tensor(targets, device=dev)
                    q[tt] ^= q[control][None, :]
                continue
            if op == "R":
                # Record actual output and phase below; never suppress errors for scoring.
                q[qs[0]] &= ~active
            elif op == "X":
                q[qs[0]] ^= active
            elif op == "CX":
                q[qs[1]] ^= q[qs[0]] & active
            elif op == "CCX":
                q[qs[2]] ^= q[qs[0]] & q[qs[1]] & active
            elif op == "HMR":
                ordinal += 1
                draw = lanes * 6364136223846793005 + (
                    (ordinal * 1442695040888963407) & MASK
                )
                draw ^= draw >> 17
                draw ^= draw << 23
                draw ^= draw >> 13
                draw &= MASK
                draw &= stream
                b = next(x for x in a[1:] if re.fullmatch(r"b\d+", x))
                bits[b] = draw
                phase ^= q[qs[0]] & draw
                q[qs[0]] = 0
            elif op == "CZ":
                phase ^= q[qs[0]] & q[qs[1]] & active
            else:
                raise ValueError((op, a))
            i += 1
        delta = words[:, k * 5 : k * 5 + 5] ^ (
            words[:, (k - 1) * 5 : k * 5] if k else torch.zeros_like(words[:, :5])
        )
        expected = target ^ delta[addr.reshape(-1)].reshape(W, 63, 5).permute(2, 0, 1)
        expected_bits = torch.stack(
            [pack((expected[j] >> b) & 1) for j in range(5) for b in range(8)]
        )
        assert torch.equal(q[:11], sel), (k, "address")
        assert not q[51:].any().item(), (k, "scratch")
        valid = ids < N

        def count(mask):
            return int(
                (((mask[:, None] >> torch.arange(63, device=dev)) & 1).bool() & valid)
                .sum()
                .item()
            )

        output_mask = (
            (q[11:51] ^ expected_bits).bitwise_or(torch.zeros_like(expected_bits)).sum(0)
            if False
            else torch.zeros(W, device=dev, dtype=torch.int64)
        )
        for row in q[11:51] ^ expected_bits:
            output_mask |= row
        phase_bad = count(phase)
        first = None
        flags = ((output_mask[:, None] >> torch.arange(63, device=dev)) & 1).bool() & valid
        if bool(flags.any().item()):
            flat = int(torch.nonzero(flags.flatten())[0].item())
            wi, bi = divmod(flat, 63)
            first = {
                "address": int(addr[wi, bi].item()),
                "target": [int(target[j, wi, bi].item()) for j in range(5)],
                "measurement_stream": int((cases[wi, bi] >> 1).item()),
                "actual": [
                    sum(
                        ((int(q[11 + j * 8 + b, wi].item()) >> bi) & 1) << b
                        for b in range(8)
                    )
                    for j in range(5)
                ],
                "expected": [int(expected[j, wi, bi].item()) for j in range(5)],
            }
        records.append(
            {
                "batch": k,
                "value_failures": count(output_mask),
                "phase_failures": phase_bad,
                "first_value_counterexample": first,
            }
        )
        print(json.dumps(records[-1]), flush=True)
    torch.mps.synchronize()
    report = {
        "device": "mps",
        "status": (
            "FAIL"
            if any(r["value_failures"] or r["phase_failures"] for r in records)
            else "PASS"
        ),
        "scope": "Exact AtakanV11 / Heliosli latest 15 ANGLE streams; all1220 addresses, both reachable incoming batches, two HMR streams. Word values, phase, selection restoration and scratch checked. This is sampled branch evidence, not an all-measurement-record proof.",
        "basis_trajectories": N * 15,
        "batches": records,
        "seconds": time.monotonic() - t,
    }
    (root / "reachable_mps_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report), flush=True)


## Credits, licences and remaining limits

- **Tanuj Khattar / Competition Host:** [official reachable-state ruling](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092). Our earlier arbitrary-target interpretation is corrected above.
- **Atakan Aldemir:** [V11, exact version 363403318](https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5?scriptVersionId=363403318), the 896Q node-7 repair, forward/replay policy consistency, reported fixed-case diagnostics and the authenticated public source archive used for this integration.
- **Mudit Chaturvedi / muchat007:** [895Q V3, exact version 363366074](https://www.kaggle.com/code/muchat007/895q-streamed-decoder-measured-node-choices?scriptVersionId=363366074), ordinary nodes 8/9, resource measurements and the unchanged 895Q candidate. The reported rare phase case is not hidden by its sampled PASS.
- **SomeJoint / Agniv Banerjee's research team:** [streamed 897Q decoder V20, 363324999](https://www.kaggle.com/code/somejoint/quantum-circuits-22-75-local-gm-897-qubit-dqi?scriptVersionId=363324999), streamed conversion/replay and arithmetic; [R8/B143/C60 chemistry and compiler, 363182923](https://www.kaggle.com/code/somejoint/quantum-circuits-22-67-local-gm-full-xor-chem?scriptVersionId=363182923), the original factor architecture, words and tree.
- **Heliosli:** [V17, exact version 363359450](https://www.kaggle.com/code/heliosli/the-dataset-is-now-public-now-enjoy?scriptVersionId=363359450), the unchanged fifteen cheaper encoded-domain angle loaders, including five later slot improvements.
- **Dmitrii Gluzdov:** [V32, exact version 363382097](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=363382097), the unchanged new numerical factor (SHA-256 `4f78b7b841745c42bb14d92e55a0ba5a9a035c752b9d7a715997395eb4729bdf`).
- **Akmal Xodarev / busyaprime**, on **mrmeows / team selenium:** [V9 arithmetic, 363321323](https://www.kaggle.com/code/busyaprime/lb-22-72431-22x-cheaper-than-google-s-circuits?scriptVersionId=363321323), the unchanged 63Q circuit, 25-AND S1 window and 172-AND main lookup. Earlier lineage includes Phuc Thai / DePie and marsyas.
- **Alex Chilton:** [latest public portfolio reviewed, V59 / 363415671](https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio?scriptVersionId=363415671), the public score comparison and earlier Schur decoder lineage; **manwithacat:** [V21 / 363419647](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made?scriptVersionId=363419647), public frontier and validation tooling. Our native checks use the original organizers' package, not the fork.
- **Kirill Tymchishin:** independent MPS reachable-state and field-algebra audits, rejected triple-column gauge search, controlled full-verifier comparisons, interpretation correction and exact-byte integration.
- **Google Quantum AI:** original benchmarks, metric, Kickmix and unchanged verifiers. The earlier source chain includes Ahmed Berat Özer, jday96314, ViberTeam / Artem1410 / anazemcev, Phuc Thai, Kozykorpesh, Michael Rustad, lightningv08 and marculera.

New integration and audit code: Apache-2.0. Original copyright and licences remain with their authors. This notebook assembles published circuit data; it does not claim a complete new source generator for every circuit or an exact proof of the full quantum algorithm. Official Public Score and local measurements remain distinct.


In [ ]:
(WORK / "CREDITS.txt").write_text(
    "## Credits, licences and remaining limits\n\n- **Tanuj Khattar / Competition Host:** [official reachable-state ruling](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092). Our earlier arbitrary-target interpretation is corrected above.\n- **Atakan Aldemir:** [V11, exact version 363403318](https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5?scriptVersionId=363403318), the 896Q node-7 repair, forward/replay policy consistency, reported fixed-case diagnostics and the authenticated public source archive used for this integration.\n- **Mudit Chaturvedi / muchat007:** [895Q V3, exact version 363366074](https://www.kaggle.com/code/muchat007/895q-streamed-decoder-measured-node-choices?scriptVersionId=363366074), ordinary nodes 8/9, resource measurements and the unchanged 895Q candidate. The reported rare phase case is not hidden by its sampled PASS.\n- **SomeJoint / Agniv Banerjee's research team:** [streamed 897Q decoder V20, 363324999](https://www.kaggle.com/code/somejoint/quantum-circuits-22-75-local-gm-897-qubit-dqi?scriptVersionId=363324999), streamed conversion/replay and arithmetic; [R8/B143/C60 chemistry and compiler, 363182923](https://www.kaggle.com/code/somejoint/quantum-circuits-22-67-local-gm-full-xor-chem?scriptVersionId=363182923), the original factor architecture, words and tree.\n- **Heliosli:** [V17, exact version 363359450](https://www.kaggle.com/code/heliosli/the-dataset-is-now-public-now-enjoy?scriptVersionId=363359450), the unchanged fifteen cheaper encoded-domain angle loaders, including five later slot improvements.\n- **Dmitrii Gluzdov:** [V32, exact version 363382097](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=363382097), the unchanged new numerical factor (SHA-256 `4f78b7b841745c42bb14d92e55a0ba5a9a035c752b9d7a715997395eb4729bdf`).\n- **Akmal Xodarev / busyaprime**, on **mrmeows / team selenium:** [V9 arithmetic, 363321323](https://www.kaggle.com/code/busyaprime/lb-22-72431-22x-cheaper-than-google-s-circuits?scriptVersionId=363321323), the unchanged 63Q circuit, 25-AND S1 window and 172-AND main lookup. Earlier lineage includes Phuc Thai / DePie and marsyas.\n- **Alex Chilton:** [latest public portfolio reviewed, V59 / 363415671](https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio?scriptVersionId=363415671), the public score comparison and earlier Schur decoder lineage; **manwithacat:** [V21 / 363419647](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made?scriptVersionId=363419647), public frontier and validation tooling. Our native checks use the original organizers' package, not the fork.\n- **Kirill Tymchishin:** independent MPS reachable-state and field-algebra audits, rejected triple-column gauge search, controlled full-verifier comparisons, interpretation correction and exact-byte integration.\n- **Google Quantum AI:** original benchmarks, metric, Kickmix and unchanged verifiers. The earlier source chain includes Ahmed Berat Özer, jday96314, ViberTeam / Artem1410 / anazemcev, Phuc Thai, Kozykorpesh, Michael Rustad, lightningv08 and marculera.\n\nNew integration and audit code: Apache-2.0. Original copyright and licences remain with their authors. This notebook assembles published circuit data; it does not claim a complete new source generator for every circuit or an exact proof of the full quantum algorithm. Official Public Score and local measurements remain distinct.\n"
)
print("Original component authors and direct version links retained in CREDITS.txt.")
